# Exercise 5: Auditing the language of generated text

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Evaluation and Benchmarks · version 0.1.0*

> Audit generated language with unrestricted LID, using human text to choose a provisional abstention threshold, and prepare a sample for fluent reviewers. Include Russian, other languages, code-switching and indeterminate short answers.

**Setup.** `pip install "stanza==1.11.0" sacrebleu torch transformers sentencepiece llama-cpp-python fasttext-wheel pyarrow requests "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this. Exercises 1 and 2 tag the **UD 2.17** Kazakh-KTB and Turkish-IMST test files with Stanza 1.11.0 over the gold words (models from `STANZA_RESOURCES_DIR`, files cached in `UD217_DIR` and checked against their SHA-256 hashes). Exercise 3 uses the X-WMT English–Turkish test set (TIL release, commit `7d988cc`), TurkicNLP's Apertium analyser and generator for Turkish and NLLB-200 distilled 600M (`NLLB_PATH`). Exercises 4 and 5 use Qwen2.5-1.5B-Instruct as an 8-bit GGUF file run with llama.cpp on the CPU (`QWEN_GGUF`, its tokeniser from `QWEN_TOKENIZER`) and GlotLID v3 (`GLOTLID_PATH`). Exercise 4 needs the Kazakh TUMLU-mini files (`TUMLU_DIR`, revision `3523de7`), checked against the hashes of the book's audit. Model outputs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

GlotLID is first run on human Kazakh news sentences of different lengths to describe label agreement by length and choose a provisional abstention threshold. It then labels 60 answers of Qwen2.5-1.5B-Instruct to Kazakh prompts of three kinds (short factual, explanatory, technical). The code prints a truncated preview of one answer per observed category; full records still need to be exported for review.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {   # UD 2.17 release files
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Kazakh-KTB/kk_ktb-ud-train.conllu": "6297597914a3f319db573573f927a6577c702f5fa00dcd653162aab46026aeaa",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
}
UD_DIR = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

def ud_path(key):
    path = UD_DIR / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    if sha256(path) != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos feats")
def read(key):
    """Sentences (lists of Word) of a UD file; multiword-token ranges and empty nodes are skipped."""
    sents, words = [], []
    for line in ud_path(key).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append(words)
            words = []
        elif not line.startswith("#"):
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[5]))
    return sents

def feats(s):
    return dict(f.split("=", 1) for f in (s or "_").split("|") if "=" in f)

def tag(lang, key):
    """Stanza 1.11 POS/feature/lemma predictions for the gold words of a UD file (cached)."""
    gold = read(key)
    path = CACHE / f"ch15-stanza-{lang}-{SHA256[key][:12]}.json"
    if not path.exists():
        import logging, stanza
        logging.getLogger("stanza").setLevel(logging.ERROR)
        nlp = stanza.Pipeline(lang, processors="tokenize,pos,lemma", tokenize_pretokenized=True, verbose=False,
                              model_dir=os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser())),
                              download_method=None)
        doc = nlp([[w.form for w in s] for s in gold])
        pred = [[[w.lemma or "_", w.upos or "_", w.feats or "_"] for w in s.words] for s in doc.sentences]
        path.write_text(json.dumps(pred, ensure_ascii=False))
    pred = json.loads(path.read_text())
    assert [len(s) for s in pred] == [len(s) for s in gold]
    return gold, [[Word(g.id, g.form, *p) for g, p in zip(gs, ps)] for gs, ps in zip(gold, pred)]

def wilson(k, n, z=1.959963984540054):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h

# ---- Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) and GlotLID, for Exercises 4 and 5 ---------------
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
_LLM = {}
def llm():
    if "m" not in _LLM:
        from llama_cpp import Llama
        _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
    return _LLM["m"]

REPLIES = CACHE / "ch15-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["value"]
def cached(key_obj, compute):
    key = hashlib.sha1(json.dumps(key_obj, ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        _CACHED[key] = compute()
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "input": key_obj, "value": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

def chat(messages, max_tokens=120):
    """Greedy reply (cached)."""
    return cached(["chat", messages, max_tokens], lambda: llm().create_chat_completion(
        messages=messages, temperature=0.0, max_tokens=max_tokens)["choices"][0]["message"]["content"].strip())

_LID = {}
def lid(text):
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

print("UD files:", ", ".join(k.split("/")[1] for k in SHA256))

In [ ]:
import csv, io
# 1. Calibration on independent human Kazakh text: KazNERD test sentences by length.
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_test.txt"
human, cur = [], []
for line in fetch(KAZNERD).read_text(encoding="utf-8").splitlines():
    if line.strip(): cur.append(line.split()[0])
    elif cur:
        human.append(" ".join(cur)); cur = []
    if len(human) == 3000: break
WORDS = re.compile(r"[^\W\d_]+")
def bucket(t):
    n = len(WORDS.findall(t)); return "1-2" if n <= 2 else "3-5" if n <= 5 else "6-10" if n <= 10 else "11+"
cal = collections.defaultdict(list)
for s in human:
    cal[bucket(s)].append(lid(s))
print("GlotLID on 3,000 human Kazakh news sentences (KazNERD), by length in words:")
for b in ("1-2", "3-5", "6-10", "11+"):
    v = cal.get(b, [])
    if v:
        ok = sum(l == "kaz_Cyrl" for l, _ in v)
        print(f"  {b:>5}: {len(v):>5} sentences, labelled kaz_Cyrl {ok / len(v):.1%}, other labels {dict(collections.Counter(l for l, _ in v if l != 'kaz_Cyrl').most_common(3))}")
probs = sorted(p for v in cal.values() for l, p in v if l == "kaz_Cyrl")
THRESH = probs[int(0.05 * len(probs))]
print(f"5th percentile of the kaz_Cyrl probability on correctly labelled human text: {THRESH:.2f} (abstain below)")

# 2. Model answers to 60 Kazakh prompts: short factual, explanatory, and technical (loanword-heavy) questions.
PROMPTS = {
 "short": ["Қазақстанның астанасы қай қала? Бір сөзбен жауап беріңіз.", "Бір жылда неше ай бар? Санмен жауап беріңіз.",
           "Ең үлкен мұхит қалай аталады? Бір сөзбен.", "Абай қай ғасырда өмір сүрген? Қысқа жауап беріңіз.",
           "Қардың түсі қандай? Бір сөзбен.", "Жеті күннің біріншісі қалай аталады? Бір сөзбен.", "Екі қосу екі неше?",
           "Қазақстанның ұлттық валютасы? Бір сөзбен.", "Күн қай жақтан шығады? Бір сөзбен.", "Алматыдағы ең биік тау шыңы? Қысқа жауап.",
           "Жылқының төлі қалай аталады? Бір сөзбен.", "Су қай температурада қайнайды? Қысқа жауап.", "Қазақ әліпбиінде неше әріп бар?",
           "Ағылшынша 'book' сөзі қазақша қалай? Бір сөзбен.", "Ең ұзын өзен қайсы? Бір сөзбен.", "Қазақстан қай материкте орналасқан? Бір сөзбен.",
           "Наурыз қай айда тойланады? Бір сөзбен.", "Бір сағатта неше минут бар? Санмен.", "Ақ пен қара араласса қандай түс шығады? Бір сөзбен.",
           "Ұлттық музыкалық аспап домбыраның неше ішегі бар? Санмен."],
 "explain": ["Неліктен жапырақтар күзде сарғаяды? Екі сөйлеммен түсіндіріңіз.", "Жаңбыр қалай пайда болады? Екі сөйлеммен.",
             "Кітап оқудың пайдасы неде? Екі сөйлеммен.", "Неге бізге ұйқы керек? Екі сөйлеммен.", "Ауылдың қаладан айырмашылығы неде? Екі сөйлеммен.",
             "Отбасы неге маңызды? Екі сөйлеммен.", "Достықтың мәні неде? Екі сөйлеммен.", "Неге суды үнемдеу керек? Екі сөйлеммен.",
             "Спорттың денсаулыққа пайдасы қандай? Екі сөйлеммен.", "Неге ағаш отырғызу керек? Екі сөйлеммен.",
             "Тіл үйренудің пайдасы неде? Екі сөйлеммен.", "Неге үлкендерді сыйлау керек? Екі сөйлеммен.", "Мектепте не үйренеміз? Екі сөйлеммен.",
             "Неге таза ауа маңызды? Екі сөйлеммен.", "Мерекелер не үшін керек? Екі сөйлеммен.", "Жазда не істеуге болады? Екі сөйлеммен.",
             "Неге тамақты дұрыс ішу керек? Екі сөйлеммен.", "Кітапхана не үшін қажет? Екі сөйлеммен.", "Табиғатты қалай қорғауға болады? Екі сөйлеммен.",
             "Бала тәрбиесінде не маңызды? Екі сөйлеммен."],
 "technical": ["Смартфонның батареясын қалай үнемдеуге болады? Екі сөйлеммен.", "Компьютерлік вирустан қалай сақтануға болады? Екі сөйлеммен.",
               "Интернет банкингті қауіпсіз пайдалану үшін не істеу керек? Екі сөйлеммен.", "Антибиотиктерді неге дәрігердің рұқсатынсыз ішуге болмайды? Екі сөйлеммен.",
               "Блокчейн технологиясы деген не? Екі сөйлеммен.", "Wi-Fi роутерді қалай баптауға болады? Екі сөйлеммен.",
               "Жасанды интеллект деген не? Екі сөйлеммен.", "Электромобильдің артықшылығы қандай? Екі сөйлеммен.",
               "Қант диабеті деген не? Екі сөйлеммен.", "Бұлттық сақтау қызметі қалай жұмыс істейді? Екі сөйлеммен.",
               "Инфляция деген не? Екі сөйлеммен.", "Квантовый компьютер деген не? Екі сөйлеммен.", "Вакцина қалай жұмыс істейді? Екі сөйлеммен.",
               "Бағдарламалау тілін қалай үйренуге болады? Екі сөйлеммен.", "Криптовалюта деген не? Екі сөйлеммен.",
               "Ипотека деген не? Екі сөйлеммен.", "Операциялық жүйені қалай жаңартуға болады? Екі сөйлеммен.", "5G желісі деген не? Екі сөйлеммен.",
               "Күн панельдері қалай жұмыс істейді? Екі сөйлеммен.", "Онлайн сабақтың кемшілігі қандай? Екі сөйлеммен."]}
SYSTEM = "Answer in Kazakh (Cyrillic script)."
answers = {g: [chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}], max_tokens=90) for q in qs] for g, qs in PROMPTS.items()}
TURKIC = {"kir_Cyrl", "tat_Cyrl", "bak_Cyrl", "kaa_Cyrl", "uzn_Cyrl", "nog_Cyrl", "alt_Cyrl", "sah_Cyrl", "chv_Cyrl", "kum_Cyrl", "krc_Cyrl", "tyv_Cyrl"}
def category(text):
    if len(WORDS.findall(text)) < 3:
        return "indeterminate (under 3 words)"
    l, p = lid(text)
    if p < THRESH:
        return "indeterminate (low confidence)"
    return "Kazakh" if l == "kaz_Cyrl" else "Russian" if l == "rus_Cyrl" else "other Turkic" if l in TURKIC else "other: " + l
print(f"\nQwen2.5-1.5B-Instruct answers (whole answer labelled; abstain below {THRESH:.2f}):")
cats = {}
for g, ans in answers.items():
    cats[g] = [category(a) for a in ans]
    print(f"  {g:<10}", dict(collections.Counter(cats[g]).most_common()))
cyr_lat = [a for g in answers for a in answers[g] if re.search(r"[A-Za-z]{3,}", a)]
print(f"answers containing a Latin-script word of 3+ letters: {len(cyr_lat)} of 60")
rng = random.Random(0)
audit = [(g, i) for g in answers for i in range(20)]; rng.shuffle(audit)
print("stratified audit sheet for a fluent reviewer (one per category):")
seen = set()
for g, i in audit:
    c = cats[g][i]
    if c in seen: continue
    seen.add(c); print(f"  [{c}] {PROMPTS[g][i]}\n     -> {answers[g][i][:160]}")

## Worked solution

Output of `experiments/ch15_exercises/run_ch15_exercises.py`:

```
GlotLID on 3,000 human Kazakh news sentences (KazNERD), by length in words:
    1-2:    34 sentences, labelled kaz_Cyrl 76.5%, other labels {'bak_Cyrl': 3, 'kir_Cyrl': 2, 'mhr_Cyrl': 2}
    3-5:   445 sentences, labelled kaz_Cyrl 96.4%, other labels {'kaa_Cyrl': 5, 'kir_Cyrl': 3, 'bak_Cyrl': 2}
   6-10:  1475 sentences, labelled kaz_Cyrl 99.8%, other labels {'rus_Cyrl': 1, 'kaa_Cyrl': 1, 'kir_Cyrl': 1}
    11+:  1046 sentences, labelled kaz_Cyrl 100.0%, other labels {}
5th percentile of the kaz_Cyrl probability on correctly labelled human text: 0.99 (abstain below)

Qwen2.5-1.5B-Instruct answers (whole answer labelled; abstain below 0.99):
  short      {'Kazakh': 13, 'indeterminate (low confidence)': 5, 'other Turkic': 2}
  explain    {'Kazakh': 9, 'other Turkic': 8, 'indeterminate (low confidence)': 3}
  technical  {'Kazakh': 10, 'other Turkic': 9, 'indeterminate (low confidence)': 1}
answers containing a Latin-script word of 3+ letters: 1 of 60
stratified audit sheet for a fluent reviewer (one per category):
  [indeterminate (low confidence)] Қазақ әліпбиінде неше әріп бар?
     -> Қазақ әліпбиінде өзгөчө әріптер бар:

1. Өзгөчө өзгөчө әріптер:
   - Өзгөчө өзгөчө әріптер
   - Өзгөчө өзгөч
  [other Turkic] Неге таза ауа маңызды? Екі сөйлеммен.
     -> Бул сөздөрдүн таза ауа маңызды:

1. "Бул сөздөрдүн таза ауа маңызды" - бул сөздөрдүн таза ауа маңызды деп аталат. Бул сөздөрдүн таза ауа
  [Kazakh] Неге тамақты дұрыс ішу керек? Екі сөйлеммен.
     -> Бұл сөздердің бірінде "тамақты" - бұл "тамақ" (тамақ - тақырып) және бірінде "дұрыс" - бұл "дұрыс" (дұрыс - дұрыс). Бұл сөздерд
```

**The news sample supports a provisional threshold.** GlotLID labels 76.5% of the 34 one- or two-word KazNERD sentences Kazakh. That share rises to 96.4% for 445 three-to-five-word sentences, 99.8% for 1,475 six-to-ten-word sentences and 100% for 1,046 longer sentences. These are agreement rates with the corpus language designation, not independently checked labels for every span.

The threshold is the fifth percentile of scores among sentences already labelled Kazakh, printed as 0.99. This is not probability calibration or a validated threshold for other languages. The sample contains no labelled non-Kazakh or mixed-language controls, and the code applies this threshold to every top label. Test false acceptances and abstentions on independent, mixed-domain data before using it operationally.

**Separate predictions from linguistic judgements.** Of 60 whole answers, 32 are labelled Kazakh, nineteen another Turkic language and nine are below the threshold. None is labelled Russian; that does not exclude Russian spans within an answer. One answer contains an ASCII-letter sequence of at least three letters, which is narrower than a complete Unicode script audit. No answer falls below the three-word cutoff.

The code prints one truncated example per observed category. It does not export full annotation records or sample every prompt group within each category. Save complete answers, sampling probabilities and a separate key; ask fluent reviewers to mark language spans, mixing, indeterminate cases and answer adequacy. A Kazakh label alone does not establish that the response answers the question.

## Your turn — extension

1. Label each sentence of the answers instead of the whole answer and report the share of Kazakh sentences per answer.
2. Add ten prompts that invite code-switching (quotations, product names) and check how the categories change.